# 03 — Final Model, Submission and Business Insights

## Objective

This notebook converts the modeling work from the previous notebooks into the final competition deliverable.

Model development is complete. The selected specification is fixed as:

- **Model:** Logistic Regression
- **Regularization:** `C=10`
- **Class weighting:** None
- **Classification threshold:** `0.43`
- **Threshold-optimized OOF F1:** `0.7712`
- **Internal test F1:** `0.7640`
- **Internal test precision:** `0.8273`
- **Internal test recall:** `0.7097`

No further model selection, hyperparameter tuning, feature engineering, or threshold optimization is performed in this notebook.

### Workflow

1. Load and verify the labeled and competition datasets
2. Reproduce the final preprocessing and model specification
3. Retrain the selected model on all cleaned labeled observations
4. Generate competition predictions and validate the submission
5. Interpret the final model
6. Translate predictive findings into business insights
7. Summarize limitations and conclusions

## 1. Imports and Configuration

The final workflow is intentionally reproducible and self-contained. The selected preprocessing pipeline and Logistic Regression specification are reconstructed explicitly before being fitted to the complete cleaned labeled dataset.

The competition test set remains separate from the labeled data and is used only for final prediction generation.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [2]:
# Project paths and final model configuration
# ---------------------------------------------------------------------------

PROJECT_ROOT = Path.cwd().parent

TRAIN_DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "conversion_data_train.csv"
)

COMPETITION_DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "conversion_data_test.csv"
)

FIGURES_PATH = PROJECT_ROOT / "outputs" / "figures"
METRICS_PATH = PROJECT_ROOT / "outputs" / "metrics"
SUBMISSIONS_PATH = PROJECT_ROOT / "outputs" / "submissions"

FIGURES_PATH.mkdir(parents=True, exist_ok=True)
METRICS_PATH.mkdir(parents=True, exist_ok=True)
SUBMISSIONS_PATH.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
FINAL_THRESHOLD = 0.43
FINAL_C = 10.0

## 2. Load and Verify the Final Datasets

The labeled training dataset and separate competition test dataset are loaded independently.

The same deterministic data-quality rule established during exploratory analysis is applied to the labeled dataset: observations with implausible ages above 80 are excluded.

No rows are removed from the competition test set because its original row count and ordering must be preserved for submission generation.

In [3]:
# Load raw datasets
# ---------------------------------------------------------------------------

train_raw = pd.read_csv(TRAIN_DATA_PATH)
competition_test = pd.read_csv(COMPETITION_DATA_PATH)

print(f"Raw labeled dataset:    {train_raw.shape}")
print(f"Competition test set:   {competition_test.shape}")

display(train_raw.head())
display(competition_test.head())

Raw labeled dataset:    (284580, 6)
Competition test set:   (31620, 5)


,country,age,new_user,source,total_pages_visited,converted
0,China,22,1,Direct,2,0
1,UK,21,1,Ads,3,0
2,Germany,20,0,Seo,14,1
3,US,23,1,Seo,3,0
4,US,28,1,Direct,3,0


,country,age,new_user,source,total_pages_visited
0,UK,28,0,Seo,16
1,UK,22,1,Direct,5
2,China,32,1,Seo,1
3,US,32,1,Ads,6
4,China,25,0,Seo,3


In [4]:
# Verify schemas
# ---------------------------------------------------------------------------

print("Labeled columns:")
print(train_raw.columns.tolist())

print("\nCompetition columns:")
print(competition_test.columns.tolist())

print("\nLabeled missing values:")
print(train_raw.isna().sum())

print("\nCompetition missing values:")
print(competition_test.isna().sum())

Labeled columns:
['country', 'age', 'new_user', 'source', 'total_pages_visited', 'converted']

Competition columns:
['country', 'age', 'new_user', 'source', 'total_pages_visited']

Labeled missing values:
country                0
age                    0
new_user               0
source                 0
total_pages_visited    0
converted              0
dtype: int64

Competition missing values:
country                0
age                    0
new_user               0
source                 0
total_pages_visited    0
dtype: int64


### 2.1 Final Data Preparation

The labeled dataset contains two observations with implausible ages above 80, previously identified during exploratory analysis. The same deterministic cleaning rule is reproduced here.

The competition test set is not filtered or reordered. Its complete set of **31,620 observations** is preserved for final prediction generation.

The target variable is separated from the five predictors, and the predictor schema is verified against the competition dataset before final model training.

In [5]:
# Apply established cleaning rule to labeled data only
# ---------------------------------------------------------------------------

train_clean = (
    train_raw
    .loc[train_raw["age"] <= 80]
    .copy()
)

removed_rows = len(train_raw) - len(train_clean)

print(f"Raw labeled observations:     {len(train_raw):,}")
print(f"Removed observations:         {removed_rows:,}")
print(f"Clean labeled observations:   {len(train_clean):,}")
print(f"Competition observations:     {len(competition_test):,}")

Raw labeled observations:     284,580
Removed observations:         2
Clean labeled observations:   284,578
Competition observations:     31,620


In [6]:
# Define final predictors and target
# ---------------------------------------------------------------------------

TARGET = "converted"

FEATURES = [
    "country",
    "age",
    "new_user",
    "source",
    "total_pages_visited",
]

X_full = train_clean[FEATURES].copy()
y_full = train_clean[TARGET].copy()

X_competition = competition_test[FEATURES].copy()

print(f"Final training features:      {X_full.shape}")
print(f"Final training target:        {y_full.shape}")
print(f"Competition features:         {X_competition.shape}")

Final training features:      (284578, 5)
Final training target:        (284578,)
Competition features:         (31620, 5)


In [7]:
# Final schema and submission-safety checks
# ---------------------------------------------------------------------------

assert X_full.columns.tolist() == FEATURES
assert X_competition.columns.tolist() == FEATURES

assert len(X_full) == len(y_full)
assert len(X_competition) == len(competition_test)

assert not X_full.isna().any().any()
assert not X_competition.isna().any().any()

assert set(y_full.unique()) == {0, 1}

print("All final data checks passed.")

All final data checks passed.


## 3. Reconstruct and Retrain the Final Model

Model development and selection were completed in the previous notebook.

The final specification is reconstructed using the same preprocessing applied during model development:

- `StandardScaler` for `age` and `total_pages_visited`;
- `OneHotEncoder(handle_unknown="ignore")` for `country` and `source`;
- passthrough for the binary `new_user` feature;
- Logistic Regression with `C=10` and no class weighting.

The classification threshold remains fixed at **0.43**.

The pipeline is now fitted on the complete cleaned labeled dataset of **284,578 observations**. This allows the final competition model to use all available labeled information after model selection and internal evaluation have been completed.

In [8]:

# Final preprocessing specification
# ---------------------------------------------------------------------------

numerical_features = [
    "age",
    "total_pages_visited",
]

categorical_features = [
    "country",
    "source",
]

binary_features = [
    "new_user",
]

final_preprocessor = ColumnTransformer(
    transformers=[
        (
            "numerical",
            StandardScaler(),
            numerical_features,
        ),
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features,
        ),
        (
            "binary",
            "passthrough",
            binary_features,
        ),
    ],
    remainder="drop",
)

In [9]:
# Final model pipeline
# ---------------------------------------------------------------------------

final_model = Pipeline(
    steps=[
        (
            "preprocessor",
            final_preprocessor,
        ),
        (
            "classifier",
            LogisticRegression(
                C=FINAL_C,
                class_weight=None,
                max_iter=1000,
                random_state=RANDOM_STATE,
            ),
        ),
    ]
)

print(final_model)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('numerical', StandardScaler(),
                                                  ['age',
                                                   'total_pages_visited']),
                                                 ('categorical',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['country', 'source']),
                                                 ('binary', 'passthrough',
                                                  ['new_user'])])),
                ('classifier',
                 LogisticRegression(C=10.0, max_iter=1000, random_state=42))])


In [10]:
# Retrain final model on all labeled observations
# ---------------------------------------------------------------------------

final_model.fit(
    X_full,
    y_full,
)

print(
    f"Final model fitted on "
    f"{len(X_full):,} labeled observations."
)

print(
    f"Fixed classification threshold: "
    f"{FINAL_THRESHOLD:.2f}"
)

Final model fitted on 284,578 labeled observations.
Fixed classification threshold: 0.43


In [11]:
# Verify fitted feature representation
# ---------------------------------------------------------------------------

final_feature_names = (
    final_model
    .named_steps["preprocessor"]
    .get_feature_names_out()
)

print(
    f"Number of transformed features: "
    f"{len(final_feature_names)}"
)

print("\nTransformed features:")
for feature in final_feature_names:
    print(f"- {feature}")

Number of transformed features: 10

Transformed features:
- numerical__age
- numerical__total_pages_visited
- categorical__country_China
- categorical__country_Germany
- categorical__country_UK
- categorical__country_US
- categorical__source_Ads
- categorical__source_Direct
- categorical__source_Seo
- binary__new_user


## 4. Competition Predictions and Submission

The final pipeline has been retrained on all cleaned labeled observations using the model specification selected before the internal test evaluation.

The fitted model is now applied to the untouched competition test set.

Predicted conversion probabilities are converted into binary predictions using the fixed classification threshold of **0.43**. The competition dataset is neither filtered nor reordered, ensuring that the generated predictions preserve the original row order.

Before creating the submission file, the predictions are checked for valid probabilities, binary output, expected row count, and plausible class distribution.

In [12]:
# Generate competition probabilities and predictions
# ---------------------------------------------------------------------------

competition_probabilities = final_model.predict_proba(
    X_competition
)[:, 1]

competition_predictions = (
    competition_probabilities >= FINAL_THRESHOLD
).astype(int)

print(
    f"Competition predictions generated: "
    f"{len(competition_predictions):,}"
)

Competition predictions generated: 31,620


In [13]:
# Competition prediction checks
# ---------------------------------------------------------------------------

assert len(competition_predictions) == len(competition_test)
assert len(competition_probabilities) == len(competition_test)

assert np.all(
    (competition_probabilities >= 0)
    & (competition_probabilities <= 1)
)

assert set(np.unique(competition_predictions)).issubset(
    {0, 1}
)

print("Prediction checks passed.")

print(
    f"\nMinimum probability: "
    f"{competition_probabilities.min():.4f}"
)
print(
    f"Maximum probability: "
    f"{competition_probabilities.max():.4f}"
)
print(
    f"Mean probability:    "
    f"{competition_probabilities.mean():.4f}"
)

print(
    f"\nPredicted conversions:     "
    f"{competition_predictions.sum():,}"
)

print(
    f"Predicted non-conversions: "
    f"{(competition_predictions == 0).sum():,}"
)

print(
    f"Predicted conversion rate: "
    f"{competition_predictions.mean():.2%}"
)

Prediction checks passed.

Minimum probability: 0.0000
Maximum probability: 1.0000
Mean probability:    0.0319

Predicted conversions:     885
Predicted non-conversions: 30,735
Predicted conversion rate: 2.80%


In [14]:
# Build competition submission
# ---------------------------------------------------------------------------

submission = pd.DataFrame(
    {
        "converted": competition_predictions
    }
)

print(f"Submission shape: {submission.shape}")

display(submission.head(10))

Submission shape: (31620, 1)


,converted
0,1
1,0
2,0
3,0
4,0
5,0
6,0
7,0
8,0
9,0


In [15]:
# Final submission validation
# ---------------------------------------------------------------------------

assert submission.shape == (len(competition_test), 1)
assert submission.columns.tolist() == ["converted"]
assert not submission.isna().any().any()
assert set(submission["converted"].unique()).issubset({0, 1})

print("Submission validation passed.")

print("\nClass counts:")
print(
    submission["converted"]
    .value_counts()
    .sort_index()
)

Submission validation passed.

Class counts:
converted
0    30735
1      885
Name: count, dtype: int64


### Save the Competition Submission

The validated binary predictions are saved as the final competition submission.

The submission contains exactly one `converted` prediction for each of the **31,620 competition observations**, preserving their original order.

In [16]:
# Save final competition submission
# ---------------------------------------------------------------------------

submission_path = (
    SUBMISSIONS_PATH
    / "conversion_predictions.csv"
)

submission.to_csv(
    submission_path,
    index=False,
)

print(f"Submission saved to: {submission_path}")

Submission saved to: c:\Users\Alex\Desktop\Jehda AI\Fullstack\CDSD Certification\3_Conversion_rate_challenge\outputs\submissions\conversion_predictions.csv


In [17]:
# Verify saved submission
# ---------------------------------------------------------------------------

saved_submission = pd.read_csv(submission_path)

assert saved_submission.shape == submission.shape
assert saved_submission.columns.tolist() == ["converted"]
assert saved_submission.equals(submission)

print("Saved submission verified successfully.")
print(f"Rows: {len(saved_submission):,}")
print(f"Columns: {saved_submission.columns.tolist()}")

display(saved_submission.head())

Saved submission verified successfully.
Rows: 31,620
Columns: ['converted']


,converted
0,1
1,0
2,0
3,0
4,0


### Submission Summary

The final competition submission contains **31,620 predictions**, matching the competition test-set size exactly.

Using the fixed classification threshold of **0.43**, the final model predicts:

- **885 conversions**
- **30,735 non-conversions**
- **2.80% predicted conversion rate**

The submission passes all structural checks: the row count is preserved, the target column contains only binary predictions, no values are missing, and the saved CSV reproduces the validated in-memory submission exactly.

The competition dataset has remained separate from model development and internal evaluation throughout the workflow.

## 5. Final Model Interpretation

The final Logistic Regression model provides a transparent view of how the available predictors are associated with conversion predictions.

Model coefficients are interpreted as **predictive associations**, not causal effects. A positive coefficient increases the model's log-odds of predicting conversion, while a negative coefficient decreases them, holding the other encoded predictors constant.

Because `age` and `total_pages_visited` are standardized during preprocessing, their coefficients represent changes associated with a one-standard-deviation increase in the corresponding feature.

Categorical variables are one-hot encoded. Their coefficients should therefore be interpreted within the model's encoded representation rather than as standalone causal effects.

In [18]:
# Extract final Logistic Regression coefficients
# ---------------------------------------------------------------------------

final_classifier = final_model.named_steps["classifier"]

coefficient_table = pd.DataFrame(
    {
        "Feature": final_feature_names,
        "Coefficient": final_classifier.coef_[0],
    }
)

coefficient_table["Abs Coefficient"] = (
    coefficient_table["Coefficient"].abs()
)

coefficient_table = (
    coefficient_table
    .sort_values(
        "Abs Coefficient",
        ascending=False,
    )
    .reset_index(drop=True)
)

coefficient_table.round(4)

,Feature,Coefficient,Abs Coefficient
0,categorical__country_China,-3.6129,3.6129
1,numerical__total_pages_visited,2.5332,2.5332
2,binary__new_user,-1.7185,1.7185
3,categorical__source_Direct,-1.4704,1.4704
4,categorical__source_Seo,-1.2819,1.2819
5,categorical__source_Ads,-1.2549,1.2549
6,numerical__age,-0.6156,0.6156
7,categorical__country_US,-0.3878,0.3878
8,categorical__country_UK,-0.0262,0.0262
9,categorical__country_Germany,0.0196,0.0196


In [19]:
# Clean transformed feature names for presentation
# ---------------------------------------------------------------------------

def clean_feature_name(feature):
    feature = (
        feature
        .replace("numerical__", "")
        .replace("categorical__", "")
        .replace("binary__", "")
    )

    feature_labels = {
        "age": "Age",
        "total_pages_visited": "Total pages visited",
        "new_user": "New user",
        "country_China": "Country: China",
        "country_Germany": "Country: Germany",
        "country_UK": "Country: UK",
        "country_US": "Country: US",
        "source_Ads": "Source: Ads",
        "source_Direct": "Source: Direct",
        "source_Seo": "Source: Seo",
    }

    return feature_labels.get(feature, feature)


coefficient_table["Display Feature"] = (
    coefficient_table["Feature"]
    .apply(clean_feature_name)
)

coefficient_table[
    ["Display Feature", "Coefficient", "Abs Coefficient"]
].round(4)

,Display Feature,Coefficient,Abs Coefficient
0,Country: China,-3.6129,3.6129
1,Total pages visited,2.5332,2.5332
2,New user,-1.7185,1.7185
3,Source: Direct,-1.4704,1.4704
4,Source: Seo,-1.2819,1.2819
5,Source: Ads,-1.2549,1.2549
6,Age,-0.6156,0.6156
7,Country: US,-0.3878,0.3878
8,Country: UK,-0.0262,0.0262
9,Country: Germany,0.0196,0.0196


### 5. Interpretation of the Final Model

The final coefficients reinforce several patterns identified during exploratory analysis.

**Total pages visited** has a strong positive coefficient (`+2.5332`). Because this variable is standardized, the coefficient represents the association of a one-standard-deviation increase in pages visited with the predicted log-odds of conversion, holding the other model inputs constant. This is consistent with the strong increase in observed conversion rates at higher engagement levels identified during EDA.

**New user** has a substantial negative coefficient (`-1.7185`). Returning visitors (`new_user = 0`) are therefore associated with a substantially higher predicted probability of conversion than new visitors, conditional on the other model features.

**Age** also has a negative coefficient (`-0.6156`), indicating that higher age is associated with lower predicted conversion probability after controlling for the other available predictors.

The largest country-specific effect is associated with **China** (`-3.6129`), which strongly reduces predicted conversion relative to otherwise comparable observations in the fitted model. This is consistent with the exceptionally low conversion rate observed for Chinese visitors during exploratory analysis.

The source coefficients are all negative in the fitted representation, with Direct showing the largest negative value. These coefficients should be interpreted jointly rather than as independent measures of channel quality because all source categories are explicitly represented by the one-hot encoding and the Logistic Regression intercept absorbs the corresponding reference level.

Similarly, the country coefficients should be interpreted as components of the complete encoded model rather than as isolated causal effects.

Overall, the strongest predictive associations are linked to **page engagement, visitor status, geography, and age**. These relationships describe how the model distinguishes converters from non-converters; they do not establish that changing any individual feature would cause conversion to increase or decrease.

## 6. Business Insights and Recommendations

The final model and exploratory analysis identify several characteristics strongly associated with conversion. These findings can help prioritize business investigation and experimentation, but they should not be interpreted as causal effects.

### 6.1 Encourage deeper on-site engagement

**Evidence:** `total_pages_visited` is the strongest positive behavioral predictor in the final model (`+2.5332` after standardization). Exploratory analysis also showed a sharp increase in observed conversion rates as the number of pages visited increased.

**Interpretation:** Users who explore more of the website are substantially more likely to convert. Page depth therefore provides a strong signal of conversion intent or engagement.

**Action / test:** Investigate which navigation paths, content, and product pages are most common among high-engagement visitors. A/B tests could evaluate whether improved recommendations, internal navigation, or relevant calls to action help users discover useful content and ultimately increase conversion.

Importantly, additional page visits should not automatically be assumed to cause conversion. They may instead reflect stronger pre-existing purchase intent.

### 6.2 Improve the new-visitor journey

**Evidence:** `new_user` has a substantial negative coefficient (`-1.7185`), and exploratory analysis showed a considerably higher conversion rate among returning visitors than new visitors.

**Interpretation:** First-time visitors appear substantially less likely to convert than users who have previously interacted with the website.

**Action / test:** Examine the first-session experience for potential friction in product discovery, trust signals, onboarding, or calls to action. Controlled experiments could test improvements targeted specifically at new visitors.

Retention and remarketing strategies may also be worth evaluating because returning visitors represent a higher-conversion segment in the observed data.

### 6.3 Investigate the exceptionally low conversion rate from China

**Evidence:** China has the strongest negative country coefficient (`-3.6129`) and exhibited an exceptionally low conversion rate during exploratory analysis compared with the other represented countries.

**Interpretation:** The size and consistency of this difference suggest that Chinese visitors may encounter market-specific barriers or represent a substantially different audience.

**Action / test:** Investigate localization, language, payment methods, pricing, delivery conditions, technical performance, traffic quality, and other market-specific factors. These explanations cannot be determined from the available dataset, so targeted diagnostic analysis should precede any major business decision.

### 6.4 Use acquisition source as a diagnostic signal, not a standalone decision rule

**Evidence:** Conversion rates differ across acquisition sources, although the differences are substantially smaller than those observed for engagement, visitor status, or China.

**Interpretation:** Traffic source contributes predictive information, but it does not appear to be the dominant factor separating converters from non-converters.

**Action / test:** Evaluate acquisition channels using richer business measures such as campaign, cost, customer value, and incremental conversion rather than reallocating marketing spend from conversion rate alone.

### 6.5 Use model scores to prioritize opportunities

The final classifier can rank visitors by predicted conversion propensity. In a real operational setting, these scores could support prioritization for appropriately designed interventions when outreach capacity is limited.

However, the model predicts **conversion propensity**, not the incremental effect of an intervention. A high predicted probability does not demonstrate that targeting a user will cause them to convert. Any targeting strategy should therefore be validated through controlled experiments and evaluated against its operational costs and benefits.

## 7. Limitations and Final Conclusion

### 7.1 Limitations

Several limitations should be considered when interpreting the model and its business implications.

**Limited feature set.** The available dataset contains only five predictors: country, age, visitor status, acquisition source, and total pages visited. Important factors such as product viewed, device, session duration, pricing, campaign details, previous purchases, and customer value are unavailable. Predictive performance is therefore constrained by the information contained in the dataset.

**Prediction does not imply causation.** The model identifies characteristics associated with conversion but cannot determine whether changing those characteristics would cause conversion behavior to change. In particular, the strong relationship between pages visited and conversion may partly reflect underlying purchase intent rather than an effect of page depth itself.

**Potential market-specific factors are unobserved.** The exceptionally low conversion rate associated with China is a strong predictive pattern, but the dataset does not explain its cause. Localization, payment options, traffic quality, technical issues, logistics, pricing, or other market-specific factors would require additional data and investigation.

**Class imbalance.** Conversion is relatively rare, representing approximately 3.2% of the cleaned labeled observations. F1-score was therefore prioritized over accuracy, and the classification threshold was optimized using out-of-fold training predictions to balance precision and recall.

**Threshold depends on the business objective.** The selected threshold of `0.43` maximizes F1 within the training-based optimization procedure. A production system with known costs for false positives and false negatives could reasonably select a different operating threshold.

**Competition predictions cannot be evaluated locally.** The separate competition dataset contains no target labels. Its predictions can therefore be structurally validated, but their predictive performance cannot be measured until an external competition score or ground truth becomes available.

### 7.2 Final Conclusion

The project developed a reproducible supervised-learning workflow for predicting website conversion while maintaining a strict separation between model development, internal evaluation, and competition prediction.

The initial Logistic Regression model proved difficult to improve meaningfully. Focused regularization tuning, polynomial feature engineering, Random Forest, and Histogram Gradient Boosting were evaluated using stratified cross-validation. Although Gradient Boosting became highly competitive, additional nonlinear complexity did not provide a meaningful advantage after equivalent threshold optimization.

The final model is therefore a **Logistic Regression classifier with `C=10`, no class weighting, and a classification threshold of `0.43`**.

Its final performance is:

- **Threshold-optimized OOF F1:** 0.7712
- **Internal test F1:** 0.7640
- **Internal test precision:** 0.8273
- **Internal test recall:** 0.7097
- **True positives:** 1,303
- **False positives:** 272
- **False negatives:** 533
- **True negatives:** 54,808

The relatively small difference between out-of-fold and internal-test performance supports the stability of the selected modeling strategy.

After model selection and evaluation were completed, the fixed pipeline was retrained on all **284,578 cleaned labeled observations** and applied to the untouched **31,620-row competition test set**. The resulting submission predicts **885 conversions**, corresponding to a predicted conversion rate of **2.80%**.

From a business perspective, the strongest predictive signals are associated with page engagement, visitor status, geography, and age. These findings provide useful directions for further investigation and controlled experimentation, particularly around the new-visitor experience, on-site engagement, and the unusually low observed conversion rate among visitors from China.

Overall, the project demonstrates that a relatively simple and interpretable model can provide strong predictive performance without unnecessary modeling complexity, while retaining a clear distinction between predictive association and causal business conclusions.

In [20]:
# Save key final model metrics
# ---------------------------------------------------------------------------

final_metrics = pd.DataFrame(
    [{
        "model": "Logistic Regression",
        "C": FINAL_C,
        "class_weight": "None",
        "threshold": FINAL_THRESHOLD,
        "oof_f1": 0.7712,
        "internal_test_f1": 0.7640,
        "internal_test_precision": 0.8273,
        "internal_test_recall": 0.7097,
        "true_negatives": 54808,
        "false_positives": 272,
        "false_negatives": 533,
        "true_positives": 1303,
        "final_training_rows": len(X_full),
        "competition_rows": len(X_competition),
        "competition_predicted_conversions": int(
            competition_predictions.sum()
        ),
        "competition_predicted_conversion_rate": (
            competition_predictions.mean()
        ),
    }]
)

metrics_path = (
    METRICS_PATH
    / "final_model_metrics.csv"
)

final_metrics.to_csv(
    metrics_path,
    index=False,
)

print(f"Final metrics saved to: {metrics_path}")

display(final_metrics.T)

Final metrics saved to: c:\Users\Alex\Desktop\Jehda AI\Fullstack\CDSD Certification\3_Conversion_rate_challenge\outputs\metrics\final_model_metrics.csv


,0
model,Logistic Regression
C,10.0
class_weight,None
threshold,0.43
oof_f1,0.7712
internal_test_f1,0.764
internal_test_precision,0.8273
internal_test_recall,0.7097
true_negatives,54808
false_positives,272
